# 01 — Data audit: what may the model actually see?

Before any model is trained, one question has to be settled: **at the moment a booking is
made, which of these 32 columns actually exist?**

The dataset was assembled *after the fact*, from a property management system. Several
columns therefore describe what happened *after* the booking — sometimes after the guest
either arrived or cancelled. A model trained on those columns will score beautifully and
be worthless in production, because on a new booking those values are unknown.

This notebook does three things:

1. Identifies every column that leaks the outcome, with evidence.
2. Documents two structural properties of the data that change how it must be split.
3. Fixes the train/test boundary used by every notebook that follows.

No model is trained here. The output is a **feature contract**.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

DATA = Path("..") / "data" / "hotel_bookings.csv.gz"
df = pd.read_csv(DATA)

print(f"rows: {len(df):,}   columns: {df.shape[1]}")
df.head(3)

rows: 119,390   columns: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## 1. The target

`is_canceled` is the target: 1 if the booking was cancelled or the guest never showed up,
0 if the guest checked out.

In [3]:
target_share = df["is_canceled"].value_counts(normalize=True).rename({0: "kept", 1: "cancelled"})
print(target_share.round(4).to_string())
print(f"\nbase rate: {df['is_canceled'].mean():.2%}")

is_canceled
kept         0.6296
cancelled    0.3704

base rate: 37.04%


37% is a workable balance. No resampling, no class weights, no SMOTE — the minority class
has 44,224 examples, which is plenty. Techniques for imbalanced data would add complexity
and distort the probabilities, and calibrated probabilities are exactly what this project
needs.

## 2. Leakage, part one: the obvious case

`reservation_status` records the final state of the booking. If it encodes the target,
it is not a feature — it *is* the target, spelled differently.

In [4]:
leak_check = pd.crosstab(df["reservation_status"], df["is_canceled"])
leak_check.columns = ["is_canceled=0", "is_canceled=1"]
print(leak_check.to_string())

                    is_canceled=0  is_canceled=1
reservation_status                              
Canceled                        0          43017
Check-Out                   75166              0
No-Show                         0           1207


Perfect separation, zero exceptions. `Canceled` and `No-Show` are always 1; `Check-Out` is
always 0. A single `if` statement on this column reproduces the target with 100% accuracy.

`reservation_status_date` goes with it: for a cancelled booking it is the date the
cancellation was entered, which cannot be known in advance.

Both are dropped.

## 3. Leakage, part two: the cases that hide

The dangerous columns are not the ones that obviously restate the outcome. They are the
ones that look like ordinary booking attributes but are in fact filled in later.

A useful test: **is any value of this column associated with a cancellation rate of
essentially zero, or essentially one?** Real predictors shift the odds. They do not
eliminate them.

In [5]:
def outcome_profile(col, clip=3):
    g = df.groupby(df[col].clip(upper=clip) if pd.api.types.is_numeric_dtype(df[col]) else df[col])
    out = g["is_canceled"].agg(bookings="size", cancel_rate="mean")
    out["cancel_rate"] = out["cancel_rate"].round(4)
    return out

for c in ["required_car_parking_spaces", "booking_changes", "total_of_special_requests"]:
    print(f"--- {c}")
    print(outcome_profile(c).to_string())
    print()

--- required_car_parking_spaces
                             bookings  cancel_rate
required_car_parking_spaces                       
0                              111974       0.3949
1                                7383       0.0000
2                                  28       0.0000
3                                   5       0.0000

--- booking_changes
                 bookings  cancel_rate
booking_changes                       
0                  101314       0.4085
1                   12701       0.1423
2                    3805       0.2013
3                    1570       0.1656

--- total_of_special_requests
                           bookings  cancel_rate
total_of_special_requests                       
0                             70318       0.4772
1                             33226       0.2202
2                             12969       0.2210
3                              2877       0.1682



### `required_car_parking_spaces` — a hard stop

7,416 bookings requested at least one parking space. **Not one of them cancelled.** Not a
low rate: zero, across three separate values of the column.

No genuine booking attribute behaves like this. Wanting a parking space cannot make
cancellation impossible. The only consistent explanation is that parking is recorded on
arrival, which means the column is populated *only for guests who showed up*.

Left in the feature set, it hands the model a free "definitely not cancelled" flag for 6%
of the data. Dropped.

### `booking_changes` — accumulates after the fact

Bookings with zero changes cancel at 40.9%; bookings with at least one change cancel at
14–20%. The direction is intuitive — someone who amends a booking is engaged with it — but
the count only grows *during the life of the booking*. At the moment of booking it is
always zero. Dropped.

### `total_of_special_requests` — kept, with a caveat

This one shifts the odds (47.7% down to 16.8%) without ever eliminating them, which is what
an honest predictor looks like. Special requests are normally captured at booking time.

It is retained, and flagged here as an assumption rather than a certainty. If the model
turns out to lean on it disproportionately, this is the first place to look.

In [6]:
df["room_changed"] = (df["assigned_room_type"] != df["reserved_room_type"]).astype(int)
print(df.groupby("room_changed")["is_canceled"].agg(bookings="size", cancel_rate="mean").round(4).to_string())

              bookings  cancel_rate
room_changed                       
0               104473       0.4156
1                14917       0.0538


### `assigned_room_type` — same problem

Bookings where the assigned room differs from the reserved room cancel at 5.4%, against
41.6% otherwise. Rooms are assigned at check-in, so a difference between the two columns is
mostly evidence that the guest arrived.

`reserved_room_type` is known at booking and is kept. `assigned_room_type` is dropped.

## 4. The deposit anomaly

This one is not leakage, but it needs to be documented before it distorts the
interpretation of the model.

In [7]:
dep = pd.crosstab(df["deposit_type"], df["is_canceled"])
dep.columns = ["kept", "cancelled"]
dep["bookings"] = dep.sum(axis=1)
dep["cancel_rate"] = (dep["cancelled"] / dep["bookings"]).round(4)
print(dep.to_string())

               kept  cancelled  bookings  cancel_rate
deposit_type                                         
No Deposit    74947      29694    104641       0.2838
Non Refund       93      14494     14587       0.9936
Refundable      126         36       162       0.2222


Bookings marked **Non Refund cancel at 99.4%** — 14,494 out of 14,587.

Read literally this is absurd: guests who paid a non-refundable deposit almost never
turned up, while guests who paid nothing turned up 72% of the time. No pricing policy
produces that.

The likelier explanation is a recording convention. In the source system, a booking that
was charged and never arrived may have been reclassified as non-refundable at the point of
cancellation, rather than at the point of sale. If so, the label partly reflects the
outcome rather than the terms agreed with the guest.

The column is **kept**, for two reasons. It is genuinely present at booking time in any
real system, and removing it would silently discard 12% of the signal. But its influence
must be watched: if the model's performance rests on this column, it is learning an
artefact of this particular database, not a property of hotel guests.

This is recorded as a limitation rather than resolved, because the dataset contains no
information that could resolve it.

## 5. Duplicate rows

The dataset has no booking identifier. That makes exact duplicates ambiguous, and there
are a great many of them.

In [8]:
dup_any = df.duplicated(keep=False)
n_extra  = df.duplicated().sum()

summary = pd.DataFrame({
    "bookings":    [dup_any.sum(), (~dup_any).sum()],
    "cancel_rate": [df.loc[dup_any, "is_canceled"].mean(), df.loc[~dup_any, "is_canceled"].mean()],
}, index=["in duplicate groups", "unique rows"]).round(4)

print(summary.to_string())
print(f"\nredundant rows if de-duplicated: {n_extra:,} ({n_extra/len(df):.1%} of the data)")

                     bookings  cancel_rate
in duplicate groups     40165       0.5839
unique rows             79225       0.2622

redundant rows if de-duplicated: 31,994 (26.8% of the data)


Rows belonging to duplicate groups cancel at **58.4%**; unique rows cancel at **26.2%**.
More than double.

Two readings are possible, and the data cannot distinguish them:

- **Real bookings.** A tour operator reserving ten rooms on the same day, for the same
  dates, through the same channel produces ten identical rows. The Groups segment cancels
  at 61.1%, which sits very close to the 58.4% observed here.
- **Data entry artefacts.** Repeated submissions of the same record.

**Decision: the duplicates are kept**, on the strength of the Groups similarity and to keep
every figure reconcilable with the Power BI report built on the same 119,390 rows.

That decision has a consequence which drives the next section. If identical rows are
allowed to fall on both sides of a random train/test split, the model can memorise a row in
training and be rewarded for recalling it in testing. The measured score would then reflect
memory rather than prediction.

## 6. Data quality

Recorded for the limitations section. None of these are corrected here — the modelling
notebook decides what to do with them, so that the decision is visible rather than buried.

In [9]:
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
guests = df["adults"] + df["children"].fillna(0) + df["babies"]

quality = pd.Series({
    "bookings with zero nights":      int((nights == 0).sum()),
    "bookings with zero guests":      int((guests == 0).sum()),
    "adr exactly zero":               int((df["adr"] == 0).sum()),
    "adr negative":                   int((df["adr"] < 0).sum()),
    "adr above 1000":                 int((df["adr"] > 1000).sum()),
})
print(quality.to_string())
print(f"\nadr range: {df['adr'].min():.2f} to {df['adr'].max():.2f}")

print("\nmissing values:")
missing = df.isna().sum()
print(missing[missing > 0].to_string())

bookings with zero nights     715
bookings with zero guests     180
adr exactly zero             1959
adr negative                    1
adr above 1000                  1

adr range: -6.38 to 5400.00

missing values:
children         4
country        488
agent        16340
company     112593


`company` is missing for 94% of bookings and `agent` for 14%. These are not errors: a
missing agent means the booking came directly rather than through an intermediary. The
absence is itself informative, so it will be encoded as its own category rather than
imputed.

## 7. Time structure and the split

The business question is forward-looking: *given the bookings on hand for a future arrival
date, how many will survive to check-in?* The evaluation has to mirror that — train on the
past, test on a future the model has never seen.

In [10]:
MONTHS = {m: i + 1 for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"])}

df["arrival_date"] = pd.to_datetime(dict(
    year =df["arrival_date_year"],
    month=df["arrival_date_month"].map(MONTHS),
    day  =df["arrival_date_day_of_month"]))

by_quarter = df.groupby(df["arrival_date"].dt.to_period("Q")).agg(
    bookings=("is_canceled", "size"), cancel_rate=("is_canceled", "mean")).round(3)
print(by_quarter.to_string())

              bookings  cancel_rate
arrival_date                       
2015Q3           11779        0.420
2015Q4           10217        0.312
2016Q1           10963        0.307
2016Q2           16198        0.375
2016Q3           15029        0.356
2016Q4           14517        0.382
2017Q1           12828        0.334
2017Q2           17621        0.435
2017Q3           10238        0.371


In [11]:
SPLIT_DATE = pd.Timestamp("2017-04-01")

train_mask = df["arrival_date"] <  SPLIT_DATE
test_mask  = df["arrival_date"] >= SPLIT_DATE

print(f"train : arrivals {df.loc[train_mask,'arrival_date'].min().date()} to "
      f"{df.loc[train_mask,'arrival_date'].max().date()}   "
      f"{train_mask.sum():>7,} bookings   cancel rate {df.loc[train_mask,'is_canceled'].mean():.3f}")
print(f"test  : arrivals {df.loc[test_mask,'arrival_date'].min().date()} to "
      f"{df.loc[test_mask,'arrival_date'].max().date()}   "
      f"{test_mask.sum():>7,} bookings   cancel rate {df.loc[test_mask,'is_canceled'].mean():.3f}")

train : arrivals 2015-07-01 to 2017-03-31    91,531 bookings   cancel rate 0.358
test  : arrivals 2017-04-01 to 2017-08-31    27,859 bookings   cancel rate 0.412


The split is drawn at **1 April 2017**: 21 months of arrivals to train on, the final 5
months held out.

Two things follow from it, and both matter.

**The test period is harder than the training period.** Cancellations run at 41.2% in the
held-out months against roughly 35.8% in training. The behaviour being modelled is
drifting. A model fitted on the earlier period will systematically under-predict the later
one — which is a real property of the problem, not a flaw in the split, and it is precisely
what the calibration step later has to correct.

**Splitting on arrival date also neutralises the duplicate problem.** Identical rows share
an arrival date by construction, so they land on the same side of the boundary. A random
split would scatter them across both.

A caveat, stated because it is a genuine weakness: bookings are assigned to train or test by
*arrival* date, not by the date they were made. A booking created in January 2017 for an
August 2017 arrival sits in the test set even though it existed before some training
bookings. Splitting on arrival date is the right choice here because the operational
question is asked per arrival night, but a stricter simulation would split on booking date
and accept a messier evaluation window.

## 8. The feature contract

The conclusion of this notebook, in a form the later notebooks import rather than restate.

In [12]:
EXCLUDE = {
    "is_canceled":                 "target",
    "reservation_status":          "leakage — encodes the target exactly",
    "reservation_status_date":     "leakage — dated after the outcome",
    "required_car_parking_spaces": "leakage — zero cancellations among 7,416 bookings; recorded on arrival",
    "assigned_room_type":          "leakage — rooms are assigned at check-in",
    "booking_changes":             "leakage — accumulates over the life of the booking",
}

contract = pd.DataFrame(
    [(c, EXCLUDE.get(c, "available at booking time"), c not in EXCLUDE) for c in df.columns
     if c not in ("arrival_date", "room_changed")],
    columns=["column", "reason", "usable"])

print(contract.to_string(index=False))
print(f"\nusable features: {contract['usable'].sum()}   excluded: {(~contract['usable']).sum()}")

                        column                                                                 reason  usable
                         hotel                                              available at booking time    True
                   is_canceled                                                                 target   False
                     lead_time                                              available at booking time    True
             arrival_date_year                                              available at booking time    True
            arrival_date_month                                              available at booking time    True
      arrival_date_week_number                                              available at booking time    True
     arrival_date_day_of_month                                              available at booking time    True
       stays_in_weekend_nights                                              available at booking time    True
          

## Summary

| Finding | Consequence |
|---|---|
| `reservation_status` reproduces the target exactly | dropped |
| `required_car_parking_spaces`: 7,416 bookings, zero cancellations | dropped — recorded on arrival |
| `assigned_room_type`, `booking_changes` populated after booking | dropped |
| `deposit_type = Non Refund` cancels at 99.4% | kept, flagged as a probable recording artefact |
| 31,994 exact duplicate rows, cancelling at more than twice the rate | kept as real group bookings |
| Test period cancels at 41.2% against 35.8% in training | drift — calibration required |
| Duplicates share an arrival date | temporal split contains them; a random split would not |

Next: `02_modelling.ipynb` builds the features, quantifies what a random split would have
cost, and trains a calibrated model.